# Moirai — Chạy với dữ liệu tài chính của bạn

Notebook này điều chỉnh demo HuggingFace để chạy với CSV tài chính cá nhân.

**Yêu cầu**: Đặt file CSV vào `data/raw/` với ít nhất 2 cột: ngày và giá trị.

In [ ]:
import sys
sys.path.append('..')

import torch
import pandas as pd
import matplotlib.pyplot as plt
from gluonts.dataset.pandas import PandasDataset
from gluonts.dataset.split import split
from huggingface_hub import hf_hub_download
from uni2ts.eval_util.plot import plot_single
from uni2ts.model.moirai import MoiraiForecast
from gluonts.evaluation import Evaluator

print(f"GPU: {torch.cuda.is_available()}")

## Cấu hình — chỉnh theo file của bạn

In [ ]:
# ===== SỬA CÁC BIẾN NÀY =====
CSV_PATH   = '../data/raw/your_data.csv'  # đường dẫn file CSV
DATE_COL   = 'Date'                        # tên cột ngày
TARGET_COL = 'Close'                       # tên cột cần dự báo
FREQ       = 'B'                           # B=business day, D=daily, W=weekly
# =============================

# Cấu hình Moirai
SIZE = "base"   # 'small' | 'base' | 'large'
PDT  = 20       # số bước dự báo (20 ngày giao dịch = ~1 tháng)
CTX  = 200      # context length
PSZ  = "auto"
BSZ  = 32
N_WINDOWS = 5   # số rolling window để đánh giá

## Load và chuẩn bị dữ liệu

In [ ]:
from src.data_loader import load_financial_dataset, make_train_test_split

ds = load_financial_dataset(CSV_PATH, TARGET_COL, DATE_COL, FREQ)
_, test_data = make_train_test_split(ds, prediction_length=PDT, n_windows=N_WINDOWS)

# Visualize
df = pd.read_csv(CSV_PATH, parse_dates=[DATE_COL]).set_index(DATE_COL)
df[[TARGET_COL]].plot(figsize=(14, 4), title=f'{TARGET_COL} — Dữ liệu đầu vào')
plt.tight_layout()
plt.show()
print(f"Tổng {len(df)} điểm | Test: {N_WINDOWS} windows x {PDT} bước")

## Load Moirai model

In [ ]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"

model = MoiraiForecast.load_from_checkpoint(
    checkpoint_path=hf_hub_download(
        repo_id=f"Salesforce/moirai-1.0-R-{SIZE}",
        filename="model.ckpt",
    ),
    prediction_length=PDT,
    context_length=CTX,
    patch_size=PSZ,
    num_samples=100,
    target_dim=1,
    feat_dynamic_real_dim=ds.num_feat_dynamic_real,
    past_feat_dynamic_real_dim=ds.num_past_feat_dynamic_real,
    map_location=device,
)
print(f"Moirai {SIZE} loaded on {device}")

## Zero-shot Inference

In [ ]:
predictor = model.create_predictor(batch_size=BSZ)

# Chạy inference
forecasts = list(predictor.predict(test_data.input))
print(f"Hoàn thành: {len(forecasts)} forecasts")

## Visualize dự báo

In [ ]:
input_it    = iter(test_data.input)
label_it    = iter(test_data.label)
forecast_it = iter(forecasts)

for i in range(min(N_WINDOWS, 3)):
    inp      = next(input_it)
    label    = next(label_it)
    forecast = next(forecast_it)

    fig, ax = plt.subplots(figsize=(12, 4))
    plot_single(inp, label, forecast, context_length=CTX,
                name=f"Window {i+1}", show_label=True, ax=ax)
    ax.set_title(f"Moirai {SIZE.upper()} Zero-Shot — {TARGET_COL} (window {i+1})")
    plt.tight_layout()
    plt.savefig(f'../results/forecast_window_{i+1}.png', dpi=150, bbox_inches='tight')
    plt.show()

## Đánh giá metric

In [ ]:
# Re-run inference để có đủ iterator cho evaluator
_, test_data_eval = make_train_test_split(ds, prediction_length=PDT, n_windows=N_WINDOWS)
forecasts_eval = list(predictor.predict(test_data_eval.input))

evaluator = Evaluator(quantiles=[0.1, 0.5, 0.9])
agg_metrics, item_metrics = evaluator(
    ts_iterator=(entry["target"] for entry in test_data_eval.label),
    fcst_iterator=iter(forecasts_eval),
)

# Bảng metric cho luận văn
metric_keys = [
    ("MAE",  "MAE"),
    ("MSE",  "MSE"),
    ("MASE", "MASE"),
    ("CRPS", "mean_wQuantileLoss"),
    ("RMSE", "RMSE"),
]
results = {}
print(f"{'Metric':10s} | {'Giá trị':>12s}")
print("-" * 25)
for label, key in metric_keys:
    val = agg_metrics.get(key)
    if val is not None:
        results[label] = round(val, 4)
        print(f"{label:10s} | {val:12.4f}")

# Lưu ra CSV
import os, pandas as pd
os.makedirs('../results', exist_ok=True)
pd.DataFrame([results]).to_csv('../results/metrics.csv', index=False)
print("\nKết quả đã lưu tại: results/metrics.csv")